In [4]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderUnavailable
import overpy
import overpass
import osm2geojson

import time
import os

import pyperclip
import geojson
import shapely.geometry as geometry
from shapely.ops import linemerge, unary_union, polygonize

import matplotlib.pyplot as plt
import matplotlib

import networkx as nx
import pandas as pd
import numpy as np
import pickle

import matplotlib.pyplot as plt
from sklearn.cluster import SpectralClustering
from sklearn.cluster import KMeans
import node2vec

import folium
import json
from shapely.geometry import mapping

from haversine import haversine

from convenient_pickle import *

In [39]:
def get_api_query(api, bbox, bicycle=True, pedestrian=True):
    if not pedestrian and not bicycle: 
        return
    #bbox = expand_bbox(bbox)
    query = "("
    if bicycle: 
        query+= f"way[highway=cycleway]{bbox};"
        query += f"way[highway=path][bicycle=designated]{bbox};"
        query += f"way[highway=path][bicycle=yes]{bbox};"
        query += f"way[footway=path][bicycle=yes]{bbox};"
    if pedestrian: 
        query += f"way[footway=sidewalk]{bbox};"
        query += f"way[footway=crossing]{bbox};"
        query += f"way[footway=path][foot=yes]{bbox};"
        query += f"way[highway=path][foot=yes]{bbox};"
        query += f"way[highway=footway]{bbox};"
    query += f"way[highway=crossing]{bbox};"
        
    query += """);
    (._;>;);
    out;"""
    result = api.query(query)
    return result

def make_a_bbox(lower, left, interval): 
    return (round(lower,2), round(left,2), round(lower+interval,2), round(left+interval,2))

def expand_bbox(bbox, margin=0.02):
    return (bbox[0]-margin, bbox[1]-margin, bbox[2]+margin, bbox[3]+margin)

def convert_to_borders(ways):
    lss = [] 
    
    for ii_w,way in enumerate(ways):
        ls_coords = []
    
        for node in way.nodes:
            ls_coords.append((node.lon,node.lat)) 
    
        lss.append(geometry.LineString(ls_coords))
    
    
    merged = linemerge([*lss]) 
    borders = unary_union(merged) # linestrings to a MultiLineString
    polygons = list(polygonize(borders))
    return merged, borders, polygons

def convert_to_dispersed_borders(ways):
    lss = [] 
    
    for ii_w,way in enumerate(ways):
        ls_coords = []
    
        for node in way.nodes:
            ls_coords.append((node.lon,node.lat)) 
    
        lss.append(geometry.LineString(ls_coords))
    
    
    merged = linemerge([*lss]) 
    return merged
    
#See how many ways each node belongs to
def collect_overlap(ways):
    outdict = dict()
    for way in ways: 
        way_nodes = way.nodes
        for node in way_nodes: 
            if node.id not in outdict.keys(): 
                outdict[node.id] = 1
            else: 
                outdict[node.id] += 1
    outdict = [(i, outdict[i]) for i in outdict.keys()]
    outdict = sorted(outdict, key = lambda x: -x[1])
    return outdict

def make_graph(ways):
    G = nx.Graph()
    
    for way in ways: 
        for node_dex in range(len(way.nodes)): 
            node = way.nodes[node_dex]
            node_id = node.id
            if G.has_node(node_id) == False:
                G.add_node(node_id)
            if node_dex > 0: 
                G.add_edge(node_id,way.nodes[node_dex-1].id)
    return G

def make_new_lcc(G, ways):
    cclist = sorted([i for i in nx.connected_components(G)], key = lambda x: -len(x))
    lcc = cclist[0]
    lcc_ways = []
    lcc_nodes = []
    for way in ways: 
        for node in way.nodes: 
            if node.id in lcc: 
                lcc_ways.append(way)
                break
    lcc_node_set = set()
    for way in lcc_ways:
        for node in way.nodes:
            lcc_node_set.add(node.id)
    return pd.Series(lcc_ways), lcc_node_set

# Serialize your merged network to GeoJSON
def quick_map(lcc_merged):
    bike_geojson = mapping(lcc_merged)
    
    # Illinois center
    m = folium.Map(location=[40.0, -89.2], zoom_start=6, tiles='CartoDB positron')
    
    # Illinois outline
    folium.GeoJson(
        "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
        name="Illinois",
        style_function=lambda f: {
            'fillColor': '#E6F1FB',
            'color': '#185FA5',
            'weight': 1.5,
            'fillOpacity': 0.25
        } if f['properties']['name'] == 'Illinois' else {
            'fillOpacity': 0,
            'color': 'none',
            'weight': 0
        }
    ).add_to(m)
    
    # Bike/pedestrian network
    folium.GeoJson(
        bike_geojson,
        name="Bike network",
        style_function=lambda f: {
            'color': '#1D9E75',
            'weight': 2,
            'opacity': 0.85
        }
    ).add_to(m)
    
    # folium.Rectangle(
    #     bounds=[[bbox[0], bbox[1]], [bbox[2], bbox[3]]],
    #     color='#E24B4A',
    #     weight=2,
    #     fill=False
    # ).add_to(m)
    
    # new_bbox = get_new_bbox(bbox, interval, 'northeast')
    
    # folium.Rectangle(
    #     bounds=[[new_bbox[0], new_bbox[1]], [new_bbox[2], new_bbox[3]]],
    #     color='#E24B4A',
    #     weight=2,
    #     fill=False
    # ).add_to(m)
    
    
    # Zoom to the network
    bounds = lcc_merged.bounds  # (minx, miny, maxx, maxy)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    
    folium.LayerControl().add_to(m)
    return m

#remember: 
#longitude becomes less negative as it goes east, more as it goes west
#latitude gets more positive as it goes north, less as it goes south
# so we have (south, west, north, east)


def shrink_bbox(use_bbox):
    #(42.10, -88.1, 42.30, -87.9)
    h_diff = use_bbox[2]-use_bbox[0]
    v_diff = use_bbox[3]-use_bbox[1]
    h_diff = h_diff * .1
    v_diff = v_diff * .1
    out_bbox = (use_bbox[0] + h_diff, use_bbox[1] + v_diff, use_bbox[2] - h_diff, use_bbox[3] - v_diff)
    return out_bbox

def find_outside_ends(nodes, use_bbox, lcc_nodes, G): 
    nodes = [node for node in nodes if node.id in lcc_nodes]
    #print(nodes)
    #bbox = shrink_bbox(use_bbox)
    bbox = use_bbox
    east = [node for node in nodes if node.lon >= bbox[3] and node.lat >= bbox[0] and node.lat <= bbox[2]]
    #east = [node for node in east if G.degree(node.id) == 1]
    southeast = [node for node in nodes if node.lon >= bbox[3] and node.lat <= bbox[0]]
    #southeast = [node for node in southeast if G.degree(node.id) == 1]
    south = [node for node in nodes if node.lon >= bbox[1] and node.lon <= bbox[3] and node.lat <= bbox[0]]
    #south = [node for node in south if G.degree(node.id) == 1]
    southwest = [node for node in nodes if node.lon <= bbox[1] and node.lat <= bbox[0]]
    #southwest = [node for node in southwest if G.degree(node.id)]
    west = [node for node in nodes if node.lon <= bbox[1] and node.lat >= bbox[0] and node.lat <= bbox[2]]
    #west = [node for node in west if G.degree(node.id) == 1]
    northwest = [node for node in nodes if node.lon <= bbox[1] and node.lat >= bbox[2]]
    #northwest = [node for node in northwest if G.degree(node.id)]
    north = [node for node in nodes if node.lat >= bbox[2] and node.lon <= bbox[3] and node.lon >= bbox[1]]
    #north = [node for node in north if G.degree(node.id) == 1]
    northeast = [node for node in nodes if node.lat >= bbox[2] and node.lon >= bbox[3]]
    #northeast = [node for node in northeast if G.degree(node.id)]
    return {'east': east, 'southeast': southeast, 'south':south, 'southwest':southwest, 'west': west,
            'northwest':northwest, 'north':north, 'northeast':northeast}

def get_new_bbox(bbox, interval, direction): 
    bottom = bbox[0]
    left = bbox[1]
    if direction == 'east': 
        return make_a_bbox(bottom, left + interval, interval)
    elif direction == 'southeast': 
        return make_a_bbox(bottom - interval, left + interval, interval)
    elif direction == 'south': 
        return make_a_bbox(bottom - interval, left, interval)
    elif direction == 'southwest': 
        return make_a_bbox(bottom - interval, left - interval, interval)
    elif direction == 'west': 
        return make_a_bbox(bottom, left - interval, interval)
    elif direction == 'northwest': 
        return make_a_bbox(bottom + interval, left - interval, interval)
    elif direction == 'north':
        return make_a_bbox(bottom + interval, left, interval)
    else: 
        return make_a_bbox(bottom+interval, left + interval, interval)

def find_connecting_bboxes(nodes, bbox, lcc_nodes, G, interval): 
    
    outside_ends = find_outside_ends(nodes, bbox, lcc_nodes, G)

    if bbox == (42.10, -88.1, 42.30, -87.9):
        print(nodes)
        print(outside_ends)
    
    outdict = dict()
    
    for key in outside_ends.keys(): 
        if outside_ends[key] != []: 
            outdict[key] = get_new_bbox(bbox, interval, key)
    return outdict

In [59]:
def scrape_the_megatrail(api, lat, lon, box_limit=40, interval=0.1,
                         used_bboxes=None, total_result_nodes=None,
                         total_result_ways=None, outbox=None): 

    interval = 0.1
    if any([i is None for i in [used_bboxes, total_result_nodes, total_result_ways]]):
        print("Starting fresh...")
        bbox = make_a_bbox(lat, lon, interval)
        okay = False
        while not okay:
            try:
                result = get_api_query(api, bbox, bicycle=True, pedestrian=True)
                okay = True
            except:
                print("Server load was too high. Waiting...")
                time.sleep(5)
        total_result_ways = result.ways
        total_result_nodes = result.nodes
        used_bboxes = [bbox]
        #save_bbox_dict = dict()
        outbox = None  # will be built below
    else:
        print("Resuming from previous run...")
        bbox = used_bboxes[-1]
        # outbox is passed in (or will be rebuilt below if None)
        bbox = used_bboxes[-1]
        
    merged, borders, polygons = convert_to_borders(total_result_ways)
    G = make_graph(total_result_ways)
    
    lcc_ways, lcc_nodes = make_new_lcc(G, total_result_ways)
    
    lcc_merged, lcc_borders, lcc_polygons = convert_to_borders(lcc_ways)
    

    bbox_dict = find_connecting_bboxes(total_result_nodes, bbox, lcc_nodes, G, interval)

    old_lcc_ways = len(lcc_ways)
    
    if outbox is None:  # <-- only initialize outbox if not restored
        outbox = [bbox_dict[key] for key in bbox_dict.keys()]
        outbox = [i for i in outbox if i not in used_bboxes]
    boxcount=1
    



    #iterate through directions and get their readings
    while outbox != [] and boxcount <= box_limit: 
        #print("I'm starting the new loop")
        #skip used bboxes
        use_bbox = outbox.pop(0)
    
        #print("I'm going through " + key)
        #print("I'm going through " + str(use_bbox))
        #try to get the api query until the request goes through
        accepted = False
        bad = 1
        while not accepted:
            # use_result = get_api_query(api, use_bbox, bicycle=True, pedestrian=True)
            # save_bbox_dict[use_bbox] = use_result
            # accepted = True
            # time.sleep(5) 
            
            try:
                use_result = get_api_query(api, use_bbox, bicycle=True, pedestrian=True)
                #save_bbox_dict[use_bbox] = use_result
                accepted = True
                time.sleep(5)
            except: 
                bad += 1
                print("That didn't work, let's try that again")
                if bad == 10:
                    print("Something went wrong. Adding use_bbox back onto the outbox.") 
                    outbox.append(use_bbox)
                    break
                time.sleep(10)
        print("Finished with the query")
        #add our new nodes and ways, then clear out duplicates
        total_result_nodes += use_result.nodes
        total_result_ways += use_result.ways
    
        use_result_nodes = []
        for way in use_result.ways:
            use_result_nodes += [i for i in way.nodes]
        
        node_dict = {n.id: n for n in total_result_nodes}
        way_dict = {w.id: w for w in total_result_ways}
        total_result_nodes = list(node_dict.values())
        total_result_ways = list(way_dict.values())
        
        G = make_graph(total_result_ways)
        
        lcc_ways, lcc_nodes = make_new_lcc(G, total_result_ways)
        #print(f"Finished {use_bbox}")
        print(f"lcc_ways is now {len(lcc_ways)} long") 
        
        
        used_bboxes.append(use_bbox)
        boxcount += 1
    
        old_lcc_ways = len(lcc_ways)
        use_bbox_nodes= [n for n in use_result_nodes if n.id in lcc_nodes]
        new_bboxes = find_connecting_bboxes(use_bbox_nodes, use_bbox, lcc_nodes, G, interval)
        new_bboxes = [new_bboxes[key] for key in new_bboxes.keys()][::-1]
        new_bboxes = [i for i in new_bboxes if i not in used_bboxes and i not in outbox]
        print(f"I'm adding {len(new_bboxes)} new bboxes")
        
        outbox += new_bboxes
    
        print(f"There are {len(outbox)} bboxes left.")
        print(f"We have {len(used_bboxes)} bboxes in total")
        print(f"{boxcount} out of {box_limit}")

    return total_result_nodes, total_result_ways, G, lcc_ways, lcc_nodes, lcc_merged, used_bboxes, outbox

In [68]:
def save_the_pickles(G, total_result_ways, total_result_nodes, used_bboxes, prefix = ''):
    current_cwd = os.getcwd()
    if prefix == '': 
        prefix='test'
    if "pickle_folder" not in os.listdir(): 
        os.mkdir("pickle_folder") 
    os.chdir("pickle_folder") 
    if prefix not in os.listdir(): 
        os.mkdir(prefix)
    os.chdir(prefix)
    convenient_pickle.dump_pickle(os.getcwd(), f'/{prefix}_graph.pkl',G)
    #convenient_pickle.dump_pickle(os.getcwd(), '/results.pkl', result)
    convenient_pickle.dump_pickle(os.getcwd(), f'/{prefix}_total_result_ways.pkl',total_result_ways)
    convenient_pickle.dump_pickle(os.getcwd(), f'/{prefix}_total_result_nodes.pkl', total_result_nodes)
    convenient_pickle.dump_pickle(os.getcwd(), f'/{prefix}_used_bboxes.pkl', used_bboxes)
    os.chdir(current_cwd)

# Begin

In [11]:
#load up APIs
user_agent = load_pickle('sorinash_agent.pkl') #You will need to provide your own user_agent for this. 
                                               #You don't need to load in a pickle, you just need a string with a title and your email address
api = overpy.Overpass(url="https://overpass-api.de/api/interpreter", user_agent = user_agent)


bicycle = True
pedestrian=True

lat = 41.8
lon = -88.3

In [62]:
total_result_nodes, total_result_ways, G, lcc_ways, lcc_nodes, lcc_merged, used_bboxes, outbox = scrape_the_megatrail(api, lat, lon, box_limit = 50)

Starting fresh...
Finished with the query
lcc_ways is now 5392 long
I'm adding 3 new bboxes
There are 5 bboxes left.
We have 2 bboxes in total
2 out of 50
That didn't work, let's try that again
That didn't work, let's try that again
Finished with the query
lcc_ways is now 18061 long
I'm adding 2 new bboxes
There are 6 bboxes left.
We have 3 bboxes in total
3 out of 50
Finished with the query
lcc_ways is now 22230 long
I'm adding 1 new bboxes
There are 6 bboxes left.
We have 4 bboxes in total
4 out of 50
Finished with the query
lcc_ways is now 25824 long
I'm adding 3 new bboxes
There are 8 bboxes left.
We have 5 bboxes in total
5 out of 50
Finished with the query
lcc_ways is now 43145 long
I'm adding 3 new bboxes
There are 10 bboxes left.
We have 6 bboxes in total
6 out of 50
Finished with the query
lcc_ways is now 45396 long
I'm adding 1 new bboxes
There are 10 bboxes left.
We have 7 bboxes in total
7 out of 50
Finished with the query
lcc_ways is now 53085 long
I'm adding 2 new bboxes


In [65]:
total_result_nodes, total_result_ways, G, lcc_ways, lcc_nodes, lcc_merged, used_bboxes, outbox = scrape_the_megatrail(api, lat, lon, box_limit = 50,used_bboxes=used_bboxes,total_result_nodes=total_result_nodes, total_result_ways=total_result_ways,outbox=outbox)

Resuming from previous run...
Finished with the query
lcc_ways is now 316721 long
I'm adding 0 new bboxes
There are 7 bboxes left.
We have 52 bboxes in total
2 out of 50
Finished with the query
lcc_ways is now 327798 long
I'm adding 1 new bboxes
There are 7 bboxes left.
We have 53 bboxes in total
3 out of 50
Finished with the query
lcc_ways is now 341498 long
I'm adding 0 new bboxes
There are 6 bboxes left.
We have 54 bboxes in total
4 out of 50
Finished with the query
lcc_ways is now 342287 long
I'm adding 0 new bboxes
There are 5 bboxes left.
We have 55 bboxes in total
5 out of 50
Finished with the query
lcc_ways is now 342287 long
I'm adding 0 new bboxes
There are 4 bboxes left.
We have 56 bboxes in total
6 out of 50
Finished with the query
lcc_ways is now 344779 long
I'm adding 1 new bboxes
There are 4 bboxes left.
We have 57 bboxes in total
7 out of 50
Finished with the query
lcc_ways is now 350739 long
I'm adding 0 new bboxes
There are 3 bboxes left.
We have 58 bboxes in total
8 

In [67]:
#lcc_merged = convert_to_dispersed_borders(lcc_ways)
#quick_map(lcc_merged)

In [73]:
os.getcwd()

'C:\\Users\\samue\\Documents\\trail_project_2026\\megatrail_scraping'

In [69]:
save_the_pickles(G, total_result_ways, total_result_nodes, used_bboxes, prefix='06_26_2026')

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x00000220718EEFE0>>
Traceback (most recent call last):
  File "C:\Users\samue\anaconda3\envs\cs7280_env_metaldata\lib\site-packages\ipykernel\ipkernel.py", line 788, in _clean_thread_parent_frames
    if phase != "start":
KeyboardInterrupt: 

KeyboardInterrupt



In [58]:
scrape_the_megatrail

<function __main__.scrape_the_megatrail(api, lat, lon, box_limit=40, interval=0.1, used_bboxes=None, total_result_nodes=None, total_result_ways=None, save_bbox_dict=None)>